# LEGO Data Analysis

This notebook explores LEGO set-release patterns and licensed themes with pandas.

**Run from the repository root** so the relative `LEGO datasets/` paths resolve. All four CSV files are loaded in the first code cell, with descriptive DataFrame names used throughout.


In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("LEGO datasets")
required_files = [
    "lego_sets.csv",
    "sets.csv",
    "themes.csv",
    "parent_themes.csv",
]
missing_files = [name for name in required_files if not (data_dir / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing dataset file(s) in {data_dir.resolve()}: {', '.join(missing_files)}. "
        "Run this notebook from the repository root."
    )

lego_sets = pd.read_csv(data_dir / "lego_sets.csv")
sets = pd.read_csv(data_dir / "sets.csv")
themes = pd.read_csv(data_dir / "themes.csv")
parent_themes = pd.read_csv(data_dir / "parent_themes.csv")

print(f"LEGO set records: {len(lego_sets):,}")
print(f"Set table records: {len(sets):,}")
print(f"Theme records: {len(themes):,}")
print(f"Parent-theme records: {len(parent_themes):,}")


In [ ]:
lego_sets.head(10)

In [ ]:
themes.head(10)

In [ ]:
sets.head(10)

In [ ]:
parent_themes.head(10)

In [ ]:
# Attach licensing metadata to each set using the parent-theme name.
merged = lego_sets.merge(
    parent_themes,
    left_on="parent_theme",
    right_on="name",
    how="left",
    suffixes=("", "_parent"),
)

merged.head()


In [ ]:
missing_set_num_count = int(merged["set_num"].isna().sum())
print(f"Rows with a missing set number: {missing_set_num_count:,}")


In [ ]:
# Keep licensed sets with a set identifier for analyses that count released sets.
licensed_df = merged.loc[
    merged["is_licensed"].eq(True) & merged["set_num"].notna()
].copy()

print(f"Licensed sets available for analysis: {len(licensed_df):,}")
licensed_df.head()


In [ ]:
star_wars_count = int(licensed_df["parent_theme"].eq("Star Wars").sum())
licensed_set_count = len(licensed_df)

if licensed_set_count == 0:
    raise ValueError("No licensed sets remain after filtering; check the source data.")

star_wars_percentage = star_wars_count / licensed_set_count * 100
print(
    f"Star Wars sets: {star_wars_count:,} of {licensed_set_count:,} "
    f"licensed sets ({star_wars_percentage:.2f}%)."
)


In [ ]:
# Count licensed sets by year and parent theme.
theme_counts = (
    licensed_df.groupby(["year", "parent_theme"], as_index=False)
    .agg(set_count=("set_num", "nunique"))
)

# Retain ties when finding the leading licensed theme(s) in each year.
annual_max = theme_counts.groupby("year")["set_count"].transform("max")
top_theme_by_year = (
    theme_counts.loc[theme_counts["set_count"].eq(annual_max)]
    .sort_values(["year", "parent_theme"])
    .reset_index(drop=True)
)
top_theme_by_year["is_star_wars"] = top_theme_by_year["parent_theme"].eq("Star Wars")

top_theme_by_year


In [ ]:
# Compare Star Wars' count with the highest licensed-theme count in each year.
all_years = pd.Index(sorted(licensed_df["year"].dropna().unique()), name="year")
star_wars_by_year = (
    licensed_df.loc[licensed_df["parent_theme"].eq("Star Wars")]
    .groupby("year")["set_num"]
    .nunique()
    .reindex(all_years, fill_value=0)
    .rename("star_wars_sets")
)
max_sets_by_year = (
    theme_counts.groupby("year")["set_count"]
    .max()
    .reindex(all_years, fill_value=0)
    .rename("most_sets_for_any_licensed_theme")
)

star_wars_comparison = pd.concat([star_wars_by_year, max_sets_by_year], axis=1).reset_index()
years_star_wars_not_top = star_wars_comparison.loc[
    star_wars_comparison["star_wars_sets"]
    < star_wars_comparison["most_sets_for_any_licensed_theme"]
].copy()

years_star_wars_not_top


In [ ]:
# Count all sets with a non-missing set number for each release year.
valid_sets = merged.loc[merged["set_num"].notna()].copy()
sets_per_year = (
    valid_sets.groupby("year", as_index=False)
    .agg(set_count=("set_num", "nunique"))
    .sort_values("year")
    .reset_index(drop=True)
)

sets_per_year
